## Minimal example of calculating NPQ-corrected fluorescence from float data

This notebook defines the function for calculating NPQ-corrected fluorescence from uncalibrated float `CHLA_FLUORESCENCE`, and applies it to some data from float ID 5906765 in the equatorial Pacific.

### Import packages, define function, read in float data

In [ ]:
import xarray as xr
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
def calculate_NPQ_fluo(ds, z_thr = 45):    
    # Calculate NPQ corrected fluorescence following Schallenberg et al. 2022 by setting values
    # to a constant above a threshold depth z_thr (default value 45m). Returns the 17-point smoothed
    # fluorescence profile, chl_smooth, and the NPQ-corrected fluorescence, fluo_npqc
    if not 'CHLA_FLUORESCENCE' in list(ds.variables):
        raise ValueError('Missing fluorescence data')
    if np.isscalar(z_thr):
        z_thr = xr.DataArray(np.full(ds['N_PROF'].shape, z_thr), coords={'N_PROF': ds['N_PROF']})
        
    # Find closest value to z_thr for each profile
    ix_z = (np.abs(ds['PRES_ADJUSTED']-z_thr)).argmin(dim='N_LEVELS')
    
    fluo_smooth = ds['CHLA_FLUORESCENCE'].rolling(N_LEVELS=17, center=True, min_periods=1).median()
    fluo_npqc = fluo_smooth.where(ds['PRES_ADJUSTED'] > z_thr, fluo_smooth.isel(N_LEVELS=ix_z))
    return (fluo_npqc, fluo_smooth)

In [ ]:
ds_fl = xr.open_dataset('data/5906765_Sprof.nc')

### Calculate and plot for the first 10 profiles

In [ ]:
ds = ds_fl.isel(N_PROF=slice(10))   
(fl_npq, fl_smooth) = calculate_NPQ_fluo(ds)

In [ ]:
fig, ax = plt.subplots()
plt.plot(fl_smooth.T, ds['PRES_ADJUSTED'].T, '-')
plt.plot(fl_npq.T, ds['PRES_ADJUSTED'].T, '--')
# match line colors
h1 = ax.get_children()
for hh in range(11, 20):
    h1[hh].set_color(h1[hh-10].get_color())
ax.set_ylim([250, 0])